# Data Lab 4 — Correlation & Root-Cause Analysis
**Certificate 04 · Data Analytics**  |  [← Back to the Lab Hub](../../index.ipynb)

Uses a shared synthetic **factory production log** — one row per part made across 3 presses, 2 shifts, 5 days. pandas, NumPy and matplotlib are pre-installed in Colab, so there is nothing to install.

## Objectives
By the end you will be able to:
- Read a scatter plot and a correlation matrix.
- Bin a variable and compute fail rate per band.
- Identify the dominant driver of defects.

## Load
We add a numeric `fail` flag (1 = Fail) so it can be correlated.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
DATA = "https://raw.githubusercontent.com/IF-JL/COEFAM-Labs/main/labs/cert04_data_analytics/data/"
df = pd.read_csv(DATA + "production_log.csv")
df["fail"] = (df["result"] == "Fail").astype(int)
df.head()

## 1 · Scatter — do temp & vibration relate to failure?

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["oven_temp_c"], df["vibration_mm_s"], c=df["fail"], cmap="coolwarm", s=6, alpha=.4)
plt.xlabel("oven_temp_c"); plt.ylabel("vibration_mm_s")
plt.title("Failures (red) cluster at high temp / vibration"); plt.colorbar(label="fail"); plt.show()

## 2 · Correlation matrix
A number from -1 to 1 for how strongly each pair moves together.

In [ ]:
cols = ["cycle_time_s", "oven_temp_c", "vibration_mm_s", "fail"]
print(df[cols].corr().round(2))

## 3 · Root cause — fail rate by temperature band

In [ ]:
df["temp_band"] = pd.cut(df["oven_temp_c"], bins=[0,200,205,210,215,999])
fr = df.groupby("temp_band", observed=True)["fail"].mean()
print(fr.round(3))
plt.figure(figsize=(7,3)); plt.bar(fr.index.astype(str), fr.values)
plt.title("Fail rate rises with temperature"); plt.ylabel("fail rate"); plt.xticks(rotation=20); plt.show()

## 4 · What are the parts actually failing for?

In [ ]:
print(df[df.fail==1]["reject_reason"].value_counts())

## Debrief
- Which variable is the strongest driver of failure — and is that correlation or causation?
- "Overheat" dominates the reject reasons. What is the concrete action for the Press_C team?